[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnrozhkov/santa-scale-challenge/blob/serverless/notebooks/01_text_to_image.ipynb)

# Text → image (Sana)

Playground for the **image** role: Nebius Sana via `adapter_for("image", settings)`.
The adapter talks OpenAI-style `images.generate` and returns PNG bytes.

**Fallback** (`SANTA_FALLBACK`, default `auto`): try Sana first; if the endpoint URL/token
is missing or the call fails, switch to OpenAI `gpt-image-1`. `off` = Sana only.
`only` = skip Sana (useful before the endpoint is RUNNING).

**Secrets.** Locally, `Settings.load()` reads `.env`. On Colab, add userdata with the
same names as `.env.example`: `IMAGE_ENDPOINT_URL`, `IMAGE_ENDPOINT_TOKEN`, and
optionally `OPENAI_API_KEY` / `SANTA_FALLBACK`. The setup cell copies those into
`os.environ`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def apply_colab_secrets() -> None:
    """Copy Colab userdata into os.environ. Secret names match .env.example."""
    try:
        from google.colab import userdata
    except ImportError:
        return
    for key in (
        "IMAGE_ENDPOINT_URL",
        "IMAGE_ENDPOINT_TOKEN",
        "VIDEO_ENDPOINT_URL",
        "VIDEO_ENDPOINT_TOKEN",
        "AUDIO_ENDPOINT_URL",
        "AUDIO_ENDPOINT_TOKEN",
        "TOKEN_FACTORY_API_KEY",
        "OPENAI_API_KEY",
        "SANTA_FALLBACK",
    ):
        try:
            value = userdata.get(key)
        except Exception:
            continue
        if value:
            os.environ[key] = str(value)


try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    apply_colab_secrets()
    repo = Path("/content/santa-scale-challenge")
    if not (repo / "pyproject.toml").is_file():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "serverless",
                "--depth",
                "1",
                "https://github.com/mnrozhkov/santa-scale-challenge.git",
                str(repo),
            ]
        )
    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])

from santa.config import Settings

settings = Settings.load()
print("SANTA_FALLBACK =", os.environ.get("SANTA_FALLBACK", "auto"))
print(
    "env present:",
    [
        k
        for k in (
            "IMAGE_ENDPOINT_URL",
            "IMAGE_ENDPOINT_TOKEN",
            "VIDEO_ENDPOINT_URL",
            "VIDEO_ENDPOINT_TOKEN",
            "AUDIO_ENDPOINT_URL",
            "AUDIO_ENDPOINT_TOKEN",
            "TOKEN_FACTORY_API_KEY",
            "OPENAI_API_KEY",
        )
        if os.environ.get(k)
    ],
)


## Parameters

Edit the next cell, or change defaults in `config/models.yaml` (loaded by `Settings`):

- **`PROMPT`** — what to draw. Or build one with `santa.prompts.image_prompt`.
- **`SEED`** — optional int on `generate(..., seed=)`. `None` leaves it to the server.
- **`SIZE`** — e.g. `1024x1024`. Lives in `roles.image.options.size`; the cell writes it
  onto `image.cfg.options` so you can experiment without editing yaml.


In [ ]:
from IPython.display import Image, display
from santa.models import adapter_for

PROMPT = (
    "A glowing red gift box on snow under a starry night, "
    "children's-book illustration, no people, no text"
)
SEED = 7  # or None
SIZE = "1024x1024"

image = adapter_for("image", settings)
image.cfg.options["size"] = SIZE
png = image.generate(PROMPT, seed=SEED)

print(image.cfg.label, "fallback=", image.used_fallback)
display(Image(data=png, format="png"))
